In [0]:
%run ./01.environment-config

In [0]:
from pyspark.sql import functions as F, DataFrame
from delta.tables import DeltaTable
from datetime import datetime, timezone
from typing import List, Dict, Any, Optional
import traceback

In [0]:
# DQ Results table — single audit trail for all checks across all tables
DQ_RESULTS_TABLE = f"{catalog_name}.{silver_schema}.data_quality_results"

DQ_RESULTS_DDL = f"""
    CREATE TABLE IF NOT EXISTS {DQ_RESULTS_TABLE} (
        run_id         STRING,
        table_name     STRING,
        batch_id       STRING,
        check_name     STRING,
        check_type     STRING,
        severity       STRING,
        passed         BOOLEAN,
        fail_count     LONG,
        total_rows     LONG,
        details        STRING,
        run_timestamp  TIMESTAMP
    )
    USING DELTA
    COMMENT 'Data Quality check results for Silver layer tables'
"""
spark.sql(DQ_RESULTS_DDL)
print(f"{DQ_RESULTS_TABLE} table created!")


In [0]:
def run_dq_checks(
    df                  : DataFrame,
    checks              : List[Dict[str, Any]],
    table_name          : str,
    batch_id            : str,
    raise_on_critical   : bool = True
) -> List[Dict[str, Any]]:
    """
    Execute a list of DQ checks against a DataFrame.

    Parameters
    ----------
    df               : DataFrame to validate
    checks           : List of check spec dicts (see check types below)
    table_name       : Target silver table name (used for logging)
    batch_id         : Current batch id (used for logging)
    raise_on_critical: If True, raises Exception when any critical check fails

    Supported check types
    ---------------------
    not_null       : { type, column, severity }
    unique         : { type, columns: [list], severity }
    min_rows       : { type, threshold, severity }
    value_range    : { type, column, min (opt), max (opt), severity }
    not_negative   : { type, column, severity }
    allowed_values : { type, column, values: [list], severity }
    referential    : { type, column, ref_table, ref_column, severity }
    """
    run_id        = f"{table_name}__{batch_id}__{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S')}"
    run_ts        = datetime.now(timezone.utc)
    results       = []
    total_rows    = df.count()

    header = f"  DQ CHECK SUITE  |  table: {table_name}  |  batch: {batch_id}"
    divider = '=' * len(header)
    print(f"\n{divider}\n{header}\n{divider}")

    for check in checks:
        check_name = check["name"]
        severity   = check.get("severity", "critical")
        check_type = check["type"]
        passed     = True
        fail_count = 0
        details    = ""

        try:
            # not_null
            if check_type == "not_null":
                col        = check["column"]
                fail_count = df.filter(F.col(col).isNull()).count()
                passed     = (fail_count == 0)
                details    = f"{fail_count:,} null values found in '{col}'"

            # unique
            elif check_type == "unique":
                cols       = check.get("columns") or [check["column"]]
                distinct   = df.dropDuplicates(cols).count()
                fail_count = total_rows - distinct
                passed     = (fail_count == 0)
                details    = f"{fail_count:,} duplicate rows on key {cols}"

            # min_rows
            elif check_type == "min_rows":
                threshold  = check["threshold"]
                passed     = (total_rows >= threshold)
                fail_count = 0 if passed else (threshold - total_rows)
                details    = f"Row count {total_rows:,} vs minimum {threshold:,}"

            # value_range
            elif check_type == "value_range":
                col     = check["column"]
                min_val = check.get("min")
                max_val = check.get("max")

                out_of_range_cond = F.lit(False)
                if min_val is not None:
                    out_of_range_cond = out_of_range_cond | (F.col(col) < min_val)
                if max_val is not None:
                    out_of_range_cond = out_of_range_cond | (F.col(col) > max_val)

                fail_count = df.filter(F.col(col).isNotNull() & out_of_range_cond).count()
                passed     = (fail_count == 0)
                details    = f"{fail_count:,} values outside [{min_val}, {max_val}] in '{col}'"

            # not_negative
            elif check_type == "not_negative":
                col        = check["column"]
                fail_count = df.filter(F.col(col).isNotNull() & (F.col(col) < 0)).count()
                passed     = (fail_count == 0)
                details    = f"{fail_count:,} negative values in '{col}'"

            # referential
            elif check_type == "referential":
                col        = check["column"]
                ref_table  = check["ref_table"]
                ref_col    = check["ref_column"]
                ref_keys   = spark.table(ref_table).select(ref_col).distinct()
                fail_count = (
                    df.filter(F.col(col).isNotNull())
                      .join(ref_keys, df[col] == ref_keys[ref_col], "left_anti")
                      .count()
                )
                passed     = (fail_count == 0)
                details    = f"{fail_count:,} values in '{col}' not found in {ref_table}.{ref_col}"

            else:
                details    = f"Unknown check type: '{check_type}' — skipped"
                passed     = True

        except Exception as e:
            passed     = False
            fail_count = -1
            details    = f"Check execution error: {str(e)}"
            traceback.print_exc()

        # print result
        if passed:
            result = "PASSED"
        elif severity == "critical":
            result = "FAILED"
        else:
            result = "WARN"
        print(f"  {result}  [{severity.upper():<8}] {check_name:<45} {details}")

        results.append({
            "run_id":        run_id,
            "table_name":    table_name,
            "batch_id":      batch_id,
            "check_name":    check_name,
            "check_type":    check_type,
            "severity":      severity,
            "passed":        passed,
            "fail_count":    int(fail_count),
            "total_rows":    int(total_rows),
            "details":       details,
            "run_timestamp": run_ts
        })

    # persist results to table
    results_df = spark.createDataFrame(results)
    results_df.write.format("delta").mode("append").saveAsTable(DQ_RESULTS_TABLE)

    ### summary ###
    total   = len(results)
    passed = sum(1 for r in results if r["passed"])
    failed = total - passed
    critical_failures = [r for r in results if not r["passed"] and r["severity"] == "critical"]

    footer = f"  SUMMARY: {passed}/{total} checks passed | {failed} failed"
    divider = '=' * len(footer)
    print(f"\n{divider}\n{footer}\n{divider}")

    if raise_on_critical and critical_failures:
        failed_names = ", ".join(r["check_name"] for r in critical_failures)
        raise Exception(
            f"DATA QUALITY CRITICAL FAILURE — table: {table_name}, batch: {batch_id}\n"
            f"Failed checks: {failed_names}\n"
            f"Results logged to: {DQ_RESULTS_TABLE}"
        )

    return results

In [0]:
# Delta CHECK constraints helper
def apply_delta_constraints(
    table_name: str,
    constraints: List[Dict[str, str]]
) -> None:
    """
    Apply Delta CHECK constraints to a table.

    constraints: list of { name: str, condition: str }
    """
    existing = {
        row["key"].replace("delta.constraints.", "")
        for row in spark.sql(f"SHOW TBLPROPERTIES {table_name}").collect()
        if row["key"].startswith("delta.constraints.")
    }

    for c in constraints:
        constraint_name = c["name"]
        condition       = c["condition"]
        if constraint_name in existing:
            print(f" Constraint already exists — skipping: {constraint_name}")
        else:
            spark.sql(
                f"ALTER TABLE {table_name} "
                f"ADD CONSTRAINT {constraint_name} CHECK ({condition})"
            )
            print(f" Constraint applied: {constraint_name} -> ({condition})")